In [1]:
!pip install qdrant-client sentence-transformers transformers torch

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [2]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

In [4]:
# 임베딩 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")

# KoBART 모델 초기화
kobart_model = AutoModelForSeq2SeqLM.from_pretrained("gogamza/kobart-base-v2")
kobart_tokenizer = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


tokenizer.json:   0%|          | 0.00/682k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


In [8]:
def search_disease_info(query_text, limit=3):
    query_vector = encoder.encode(query_text).tolist()

    search_results = client.search(
        collection_name='son5',
        query_vector=query_vector,
        limit=limit
    )

    results = []
    for hit in search_results:
        result = {
            "질병": hit.payload.get("질병", "N/A"),
            "답변": hit.payload.get("답변", "N/A"),
            "유사도 점수": hit.score
        }
        results.append(result)

    return results

In [9]:
def generate_response(query, context):
    input_text = f"질문: {query}\n컨텍스트: {context}\n답변:"
    input_ids = kobart_tokenizer.encode(input_text, return_tensors="pt", max_length=1024, truncation=True)

    with torch.no_grad():
        output = kobart_model.generate(
            input_ids,
            max_length=512,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            top_k=50,
            top_p=0.92,
            temperature=0.7,
            do_sample=True,
            early_stopping=True,
        )

    response = kobart_tokenizer.decode(output[0], skip_special_tokens=True)
    return response.strip()


In [10]:
def advanced_rag_search():
    while True:
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ")
        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        results = search_disease_info(query)
        if results:
            best_match = results[0]  # 가장 유사도가 높은 결과 선택
            context = best_match['답변']

            print(f"\n'{query}'에 대한 검색 결과:\n")
            print(f"관련 질병: {best_match['질병']}")
            print(f"유사도 점수: {best_match['유사도 점수']:.4f}")
            print("원본 답변:")
            print(context)
            print("\nKoBART 생성 답변:")
            generated_response = generate_response(query, context)
            print(generated_response)
        else:
            print("검색 결과가 없습니다.")
        print("-" * 50)

# 실행
advanced_rag_search()

질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

관련 질병: 아킬레스 건염
유사도 점수: 0.8145
원본 답변:
아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로, 가능한 빠른 시일 내에 증상을 인지하고 의사의 도움을 받는 것이 중요합니다.

KoBART 생성 답변:


/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:588: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


질문: 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나
컨텍스트: 아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아k레스 걸염 은 발 목 부근에 발생하는 많이 발발 부근이 발생하는염증이 질환이 잘 잘 많이 많이 있습니다. 이 발들 주변에서 통증과 부종을 동반하는 통 증상이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킨레스 이 아 될레스 다른 부분 부분에서 염증이 발생하면 발순 주변에 강한 통리가 생길 수 있습니다. 또한, 아킬로레스 양염을 종아리로 통지가 전해질 수 있어 발목을 부분을 걷거나 달리는 동안 통위가 느껴질 때 있습니다. 만약 아릴레스 진을 받을 수 있는 발등 부분을 걸거나 뛰는 동안통증이 느껴 질 것이 중요합니다. 아왈레스 편염이 다양한 증상을 동반 하는 염장성질환입니다. 만약 발집 주변에서 통상이 나타나면 의사의 진료를 잘 이상 이상, 의 의문의 진이 이상 반드시 발길 주변에서통 증상이 나타나 지면 임 임 하 하하게 잘가가 만약, 발주 주변에서 걸증이 나타나 있으면 의 등의 통 문제가 나타나, 가능한 빠른 빠른 시일 내에 증 증진을 인지하고 의인의 도움을 받는  있으므로, 가능 빠른 빨리 내에증상을 인지 하고 의정의 도움을 필요한하고 인사의 도움을 연결하고 제사의 의치의 도움을 받 것이 추천변변:답변들 말씀변이 증이 전해 질 수 있어, 발아 부분을 들거나 걷는거나 달리 걷는 거 달리는 같은 통 시간이 달리는 계속해서 통가가 지속되는 계속해서 계속 계속되는 경우 의자의 진질을 진제를 받는 것이 매우합니다.  아키는레스 거염, 다양한 다양한 수상을 지속하는 염 염가성성 원인입니다. 예를 발어 등 잘잘 잘 하는 많이  있는 염증을 동반 있는염장 질환입니다. 있다. 만약발목 주변에서  통 소리가 나타나지 통 아이가 나타나 안 의가 통일이 나타나하면 의수의 진 진료 진리를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로 매우 심각한 심각한합병증을 유발할 한 수므로, 빠른 가